In [1]:
import pdfplumber
from PIL import Image
from datasets import Dataset
import torch
from transformers import (
    AutoProcessor,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    LayoutLMv3Processor,
    LayoutLMv3ForTokenClassification,
)

/Users/naveen1.mathur/Desktop/x/sftc/_learning/ml/mlp/ml-projects/_env_05_OCR_DOCUMENT_PARSER/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
pdf_path = "./files/invoice.pdf"
dataset_list = []

with pdfplumber.open(pdf_path) as pdf:
    for page_number, page in enumerate(pdf.pages):
        # Convert page to PIL image
        img = page.to_image(resolution=350).original

        # Extract words + bbox
        words = []
        bboxes = []
        for word in page.extract_words():
            words.append(word["text"])
            # Normalize bbox to 0-1000
            x0 = int(word["x0"] / page.width * 1000)
            y0 = int(word["top"] / page.height * 1000)
            x1 = int(word["x1"] / page.width * 1000)
            y1 = int(word["bottom"] / page.height * 1000)
            bboxes.append([x0, y0, x1, y1])

        # Labels (0=O for unsupervised; replace with manual annotations for NER)
        labels = [0] * len(words)

        dataset_list.append(
            {"image": img, "words": words, "bboxes": bboxes, "labels": labels}
        )

dataset = Dataset.from_list(dataset_list)

In [3]:
import fitz  # PyMuPDF

pdf_path = "./files/invoice.pdf"
output_pdf_path = "./files/output_with_boxes.pdf"

# Open the original PDF
doc = fitz.open(pdf_path)

for page_idx, data in enumerate(dataset_list):
    page = doc[page_idx]
    words = data["words"]
    bboxes = data["bboxes"]

    for bbox in bboxes:
        # Denormalize bbox back to PDF coordinates
        x0 = bbox[0] / 1000 * page.rect.width
        y0 = bbox[1] / 1000 * page.rect.height
        x1 = bbox[2] / 1000 * page.rect.width
        y1 = bbox[3] / 1000 * page.rect.height

        # Draw rectangle
        rect = fitz.Rect(x0, y0, x1, y1)
        page.draw_rect(rect, color=(1, 0, 0), width=1)  # red box

# Save the PDF
doc.save(output_pdf_path)
doc.close()

print(f"Saved output PDF with boxes: {output_pdf_path}")

Saved output PDF with boxes: ./files/output_with_boxes.pdf


In [4]:
model_name = "microsoft/layoutlmv3-base"

# ⚠ apply_ocr=False here
processor = AutoProcessor.from_pretrained(model_name, apply_ocr=False)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=3,  # adjust for your task
)

Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 40.62it/s]
Some weights of LayoutLMv3ForTokenClassification were not initialized from the model checkpoint at microsoft/layoutlmv3-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
def preprocess(batch):
    encoded = processor(
        images=batch["image"],
        text=batch["words"],
        boxes=batch["bboxes"],
        word_labels=batch["labels"],
        return_tensors="pt",
        padding="max_length",
        truncation=True,
    )
    return encoded


encoded_dataset = dataset.map(
    preprocess, batched=True, remove_columns=dataset.column_names
)

Map: 100%|██████████| 2/2 [00:00<00:00,  8.30 examples/s]


In [6]:
args = TrainingArguments(
    output_dir="./layoutlmv3-finetune",
    per_device_train_batch_size=1,  # for single PDF
    learning_rate=5e-5,
    weight_decay=0.01,
    num_train_epochs=5,
    logging_steps=1,
    save_steps=10,
    # fp16=torch.backends.mps.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=encoded_dataset,
)

In [7]:
# trainer.train()
# model.save_pretrained("./layoutlmv3-finetune")
# processor.save_pretrained("./layoutlmv3-finetune")

In [8]:
model_dir = "./layoutlmv3-finetune"
processor = AutoProcessor.from_pretrained(model_dir, apply_ocr=False)
model = AutoModelForTokenClassification.from_pretrained(model_dir)

In [9]:
model.eval()  # set to evaluation mode
pdf_path = "./files/invoice.pdf"
# pdf_path = "./files/a_01.pdf"

In [10]:
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[0]  # first page
    img = page.to_image(resolution=300).original

    words = []
    bboxes = []
    for word in page.extract_words():
        words.append(word["text"])
        x0 = int(word["x0"] / page.width * 1000)
        y0 = int(word["top"] / page.height * 1000)
        x1 = int(word["x1"] / page.width * 1000)
        y1 = int(word["bottom"] / page.height * 1000)
        bboxes.append([x0, y0, x1, y1])

In [11]:
inputs = processor(
    images=img,
    text=words,
    boxes=bboxes,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
)

In [12]:
with torch.no_grad():
    outputs = model(**inputs)

/Users/naveen1.mathur/Desktop/x/sftc/_learning/ml/mlp/ml-projects/_env_05_OCR_DOCUMENT_PARSER/lib/python3.9/site-packages/transformers/modeling_utils.py:1742: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


In [13]:
predictions = torch.argmax(outputs.logits, dim=-1).squeeze().tolist()

In [14]:
id = 1
labels = []
for word, label_id, bbox in zip(words, predictions, bboxes):
    print(f"id: [{id}] -> {word} | Label: {label_id} | BBox: {bbox}")
    labels.append(word)
    id += 1

id: [1] -> Reliance | Label: 0 | BBox: [396, 48, 474, 63]
id: [2] -> Retail | Label: 0 | BBox: [479, 48, 530, 63]
id: [3] -> Limited | Label: 0 | BBox: [536, 48, 603, 63]
id: [4] -> SCO | Label: 0 | BBox: [280, 111, 318, 125]
id: [5] -> No.123 | Label: 0 | BBox: [323, 111, 379, 125]
id: [6] -> - | Label: 0 | BBox: [383, 111, 389, 125]
id: [7] -> 124, | Label: 0 | BBox: [394, 111, 428, 125]
id: [8] -> Sector | Label: 0 | BBox: [433, 111, 483, 125]
id: [9] -> - | Label: 0 | BBox: [487, 111, 493, 125]
id: [10] -> 17C | Label: 0 | BBox: [498, 111, 530, 125]
id: [11] -> Chandigarh | Label: 0 | BBox: [535, 111, 624, 125]
id: [12] -> Chandigarh | Label: 0 | BBox: [629, 111, 719, 125]
id: [13] -> Chandigarh | Label: 0 | BBox: [423, 128, 513, 142]
id: [14] -> 160017 | Label: 0 | BBox: [518, 128, 576, 142]
id: [15] -> (Original | Label: 0 | BBox: [415, 197, 481, 211]
id: [16] -> for | Label: 0 | BBox: [485, 197, 506, 211]
id: [17] -> Recipient) | Label: 0 | BBox: [511, 197, 588, 211]
id: [18] ->

In [15]:
labels

['Reliance',
 'Retail',
 'Limited',
 'SCO',
 'No.123',
 '-',
 '124,',
 'Sector',
 '-',
 '17C',
 'Chandigarh',
 'Chandigarh',
 'Chandigarh',
 '160017',
 '(Original',
 'for',
 'Recipient)',
 'Tax',
 'Invoice',
 'Invoice',
 'No',
 ':',
 'A4R26R9999256622',
 'Invoice/Payment',
 'Date',
 '&',
 'Time',
 ':',
 '06',
 'Aug,2025',
 '20:39:27',
 'PAN',
 'No',
 ':',
 'AABCR1718E',
 'GST',
 'No',
 ':',
 '04AABCR1718E1ZX',
 'Order',
 'Ref.',
 'No.',
 ':',
 'TB00004DEE6I',
 'Payment',
 'Ref.',
 'No.',
 ':',
 '109367559603',
 'Mode',
 'of',
 'Payment',
 ':',
 'UPI',
 'Customer',
 'Name',
 ':',
 'Devka',
 'Trehan',
 'Place',
 'of',
 'Supply',
 ':',
 '04',
 'Chandigarh',
 'Customer',
 'Address',
 ':HOUSE',
 'NUMBER',
 '5421/2,',
 'Chandigarh,',
 'Jio',
 'Number',
 ':',
 '1723567615',
 'MODERN',
 'HOUSING',
 'COMPLEX,',
 'NEAR',
 'MANIMAJRA',
 'MARKET,',
 '4,',
 'Chandigarh,',
 '160101',
 'Sr.',
 'Taxable',
 'Item',
 'Name',
 'HSN/SAC',
 'Qty',
 'MRP/Unit(',
 '`)',
 'Discount(',
 '`)',
 'No.',
 'Amount(

In [16]:
labels[98]

'JioFiber_3M_1197'